In [2]:
!pip install tensorflow


   ---------------------------------------- 0.0/351.2 MB ? eta -:--:--
   ---------------------------------------- 3.7/351.2 MB 20.6 MB/s eta 0:00:17
    --------------------------------------- 8.1/351.2 MB 20.4 MB/s eta 0:00:17
   - -------------------------------------- 11.8/351.2 MB 19.3 MB/s eta 0:00:18
   - -------------------------------------- 14.7/351.2 MB 17.9 MB/s eta 0:00:19
   -- ------------------------------------- 18.4/351.2 MB 17.7 MB/s eta 0:00:19
   -- ------------------------------------- 23.3/351.2 MB 18.7 MB/s eta 0:00:18
   --- ------------------------------------ 31.2/351.2 MB 21.5 MB/s eta 0:00:15
   ---- ----------------------------------- 39.1/351.2 MB 23.4 MB/s eta 0:00:14
   ----- ---------------------------------- 46.7/351.2 MB 24.9 MB/s eta 0:00:13
   ------ --------------------------------- 53.2/351.2 MB 26.0 MB/s eta 0:00:12
   ------ --------------------------------- 53.2/351.2 MB 26.0 MB/s eta 0:00:12
   ------ --------------------------------- 53.2/35

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.51.0 requires protobuf<7,>=3.20, but you have protobuf 7.34.1 which is incompatible.


In [3]:
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [4]:
# ===============================
# 1. Load Processed Data
# ===============================
df = pd.read_csv("processed_diabetes.csv")

print("Dataset loaded:", df.shape)


Dataset loaded: (768, 18)


In [5]:
# ===============================
# 2. Define Features & Target
# ===============================
X = df.drop('Outcome', axis=1)
y = df['Outcome']

print("Feature columns:", X.columns.tolist())

Feature columns: ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'BMI_Category', 'Age_Group', 'Glucose_Level', 'BMI_Age', 'Glucose_BMI', 'Insulin_Glucose_Ratio', 'Pregnancy_Age_Ratio', 'Pregnancy_Risk', 'Insulin_log']


In [6]:
# ===============================
# 3. Train-Test Split
# ===============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [7]:
# ===============================
# 4. Feature Scaling (IMPORTANT)
# ===============================
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Save scaler for Streamlit
joblib.dump(scaler, "scaler.pkl")

['scaler.pkl']

In [8]:
# ===============================
# 5. Build Improved ANN Model
# ===============================
model = Sequential()

# Input Layer + Hidden Layers
model.add(Dense(32, activation='relu', input_dim=X_train.shape[1]))
model.add(Dropout(0.3))  # prevent overfitting

model.add(Dense(16, activation='relu'))
model.add(Dropout(0.2))

model.add(Dense(8, activation='relu'))

# Output Layer
model.add(Dense(1, activation='sigmoid'))

# Compile model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

C:\Users\user\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 32)                  │             576 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 16)                  │             528 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 16)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 8)                   │             136 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 1)                   │               9 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,249 (4.88 KB)

 Trainable params: 1,249 (4.88 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
# ===============================
# 6. Training with Early Stopping
# ===============================
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=16,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.5703 - loss: 0.6739 - val_accuracy: 0.6748 - val_loss: 0.6248
Epoch 2/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6762 - loss: 0.6188 - val_accuracy: 0.7073 - val_loss: 0.5752
Epoch 3/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7026 - loss: 0.5731 - val_accuracy: 0.7317 - val_loss: 0.5392
Epoch 4/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6986 - loss: 0.5414 - val_accuracy: 0.7398 - val_loss: 0.5115
Epoch 5/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7088 - loss: 0.5399 - val_accuracy: 0.7398 - val_loss: 0.5033
Epoch 6/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7393 - loss: 0.5037 - val_accuracy: 0.7642 - val_loss: 0.4934
Epoch 7/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7515 - loss: 0.5007 - val_accuracy: 0.7561 - val_loss: 0.4842
Epoch 8/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7841 - loss: 0.4903 - val_accuracy: 0.7886 - v

In [10]:
# ===============================
# 7. Model Evaluation
# ===============================
y_pred = (model.predict(X_test) > 0.5).astype(int)

print("\nModel Evaluation:")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step

Model Evaluation:
Accuracy: 0.7272727272727273

Classification Report:
               precision    recall  f1-score   support

           0       0.79      0.79      0.79       100
           1       0.61      0.61      0.61        54

    accuracy                           0.73       154
   macro avg       0.70      0.70      0.70       154
weighted avg       0.73      0.73      0.73       154


Confusion Matrix:
 [[79 21]
 [21 33]]


In [ ]:
# ===============================
# 8. Save Model
# ===============================
model.save("diabetes_ann_model.h5")

print("\nModel and scaler saved successfully!")